# COGS 188 — Project Proposal
## University Course Timetabling with Search and Evolutionary Optimization

**Group members:** Derrick Chen, Zhen Wang, and Daira Merino  
**Project type:** Implementation  
**Repository:** [zhc066/COGS188_group](https://github.com/zhc066/COGS188_group)

# Abstract

We propose to compare AI search methods for constructing university course timetables. The task is to assign weekly lecture meetings to rooms and time slots while avoiding room, instructor, and student-cohort conflicts. Among feasible schedules, we will minimize penalties for inconvenient gaps, undesirable meeting times, and changes of room within a course. We will generate reproducible scheduling instances with different sizes and levels of resource contention. A greedy scheduler will provide a baseline, and we will implement simulated annealing and a genetic algorithm to investigate whether broader exploration improves schedule quality. We will evaluate feasibility rate, preference penalties, runtime, and consistency across random seeds under equal computation budgets. Our central question is how the methods' tradeoffs change as scheduling becomes more constrained. The intended outcome is a reproducible comparison and timetable visualization, rather than a production scheduling system.

# Background

Every term, universities must place hundreds of course meetings into a limited set of rooms and time slots, and the resulting timetable shapes students' days, instructors' workloads, and how efficiently rooms are used. Automated timetabling has been studied for decades in operations research and AI; surveys distinguish school, course, and examination timetabling and catalog the hard and soft constraints each involves<a name="schaerf"></a>[<sup>[1]</sup>](#schaerfnote)<a name="burke"></a>[<sup>[2]</sup>](#burkenote). Even simplified timetabling problems are NP-complete<a name="even"></a>[<sup>[3]</sup>](#evennote), so practical systems rely on heuristic search rather than exhaustive enumeration.

The second International Timetabling Competition (ITC-2007) included a curriculum-based course timetabling track in which lectures are assigned to periods and rooms, with conflicts determined by instructors and curricula. One solver applied to all three ITC-2007 tracks first constructs a complete timetable and then improves it with hill climbing, great deluge, and optionally simulated annealing<a name="muller"></a>[<sup>[4]</sup>](#mullernote). We use this formulation as background for a simplified, explicitly documented formulation; our scores will not be presented as official ITC benchmark scores.

Local search represents a candidate timetable as a complete assignment and improves it through small changes<a name="aima"></a>[<sup>[5]</sup>](#aimanote). The min-conflicts heuristic showed that repairing a complete but flawed assignment can solve large constraint satisfaction and scheduling problems, and it was used to schedule Hubble Space Telescope observations<a name="minton"></a>[<sup>[6]</sup>](#mintonnote). Simulated annealing also accepts some worsening moves, with a probability that falls as a temperature parameter decreases<a name="kirkpatrick"></a>[<sup>[7]</sup>](#kirkpatricknote), and was applied to school timetabling early on<a name="abramson"></a>[<sup>[8]</sup>](#abramsonnote). Genetic algorithms instead search with a population of candidates using selection, crossover, and mutation<a name="holland"></a>[<sup>[9]</sup>](#hollandnote)[<sup>[5]</sup>](#aimanote). A survey of metaheuristics for university timetabling groups methods by how they handle hard constraints: optimizing one weighted objective, first reaching feasibility and then optimizing preferences, or relaxing some constraints<a name="lewis"></a>[<sup>[10]</sup>](#lewisnote). Our objective (defined below) follows the weighted approach while weighting feasibility so heavily that it always takes priority.

Benchmark studies typically report each method's best tuned result on fixed instances. We instead vary resource contention systematically under equal time budgets, which lets us ask how the trade-off between improving one candidate (simulated annealing) and maintaining a population (genetic algorithm) changes as problems become tighter. Our hypothesis is that both will improve preference satisfaction over a single greedy pass, but that the genetic algorithm's population overhead may make it less competitive under short time limits. This is a hypothesis to test, not an assumed result.

# Problem Statement

**Research question:** Under a fixed computation budget, which method most reliably produces feasible, low-penalty course timetables as resource contention increases?

An instance contains courses, required weekly meetings, instructors, student cohorts, rooms, and available periods. Each meeting lasts one period. Each course has one instructor and may belong to multiple cohorts. A schedule assigns every required meeting to exactly one room and period.

**Hard constraints**
- A room cannot host two meetings in the same period.
- An instructor cannot teach two meetings in the same period.
- Courses sharing a cohort cannot meet simultaneously.
- Meetings must respect instructor availability and room capacity.
- Multiple meetings of a course must occur on different days.

**Soft constraints**
- Reduce idle periods between a cohort's first and last meeting of each day.
- Avoid designated undesirable periods.
- Keep each course in as few different rooms as possible.

Let H be the total hard-constraint violation count. Define S = G + U + R, where G counts cohort idle periods, U counts meetings in undesirable periods, and R sums max(0, distinct rooms used − 1) across courses. We will document exact counting conventions and keep these unit weights fixed in the primary comparison.

Feasibility takes priority over preferences. Search will minimize E = M H + S, with M greater than a computed upper bound on S for each instance. Thus, reducing one hard violation always outweighs any possible preference improvement. Final results will distinguish feasible and infeasible outputs.

The problem is non-trivial because assignments interact across several shared resources, and even simplified timetabling is NP-complete[<sup>[3]</sup>](#evennote). For L meetings, P periods, and R rooms, the unconstrained assignment space contains (P R)^L possibilities. Our largest planned instances have about 250 meetings, 30 periods, and 10–21 rooms, giving at least 300^250 ≈ 10^619 assignments, so enumeration is impossible.

# Data

Our primary inputs will be **synthetic scheduling instances**, not private university records. An observation is an entire scheduling instance, with linked course, meeting, room, and availability tables.

We plan to generate 1,200 instances: 200 for development and parameter selection and 1,000 held out for evaluation. Proposed instance sizes are 20, 40, 60, and 100 courses, with five teaching days of six periods each (P = 30). Each course will require two or three weekly meetings. We define **resource contention** as room utilization u = L / (R P), the fraction of all room-periods that must be filled, and choose the number of rooms R to target three levels: low (u ≈ 0.4), medium (u ≈ 0.6), and high (u ≈ 0.8). Within each level we will also vary cohort overlap and instructor availability. Development and test instances will be split approximately evenly across the 4 × 3 size–contention cells (about 17 development and 83 test instances per cell).

Recorded fields will include instance ID, generation seed, course ID, instructor ID, cohort IDs, enrollment, required meeting count, meeting ID, room ID, room capacity, day, period, availability, and undesirable-period indicators. Lists and availability masks will be encoded explicitly in JSON; generated configurations and seeds will be versioned.

To avoid unintentionally impossible primary cases, we will first construct a valid hidden timetable and derive compatible availability, room capacities, and conflicts from it. Solvers will receive only the resulting instance. The hidden timetable will verify feasibility, not serve as an optimum or solver initialization. Separate seeds will produce development and test instances.

We will validate identifiers, references, required meetings, capacities, and the hidden timetable with an independent constraint checker. We will report distributions of instance size and resource contention. Because this is an optimization project rather than supervised prediction, the instances are search problems rather than labeled training examples. We request approval for this formulation and its synthetic evaluation design; synthetic realism remains a limitation.

# Proposed Solution

We will implement a shared Python representation, objective function, and independent schedule validator, then compare:

1. **Greedy baseline.** Order meetings by the number of conflicts and available assignments, placing the most constrained first (the minimum-remaining-values idea from constraint satisfaction[<sup>[5]</sup>](#aimanote)). Choose a feasible placement with the smallest added preference penalty. If no feasible placement exists, use the placement with the smallest added E and record the resulting violations. Use fixed tie-breaking.
2. **Simulated annealing.** Begin from the greedy timetable. Propose moving one meeting or swapping two meetings. Accept improvements and accept a worsening change of size ΔE with probability exp(−ΔE/T)[<sup>[7]</sup>](#kirkpatricknote). Gradually decrease T and retain the best candidate encountered. As an ablation, we will also run hill climbing (the same moves with T = 0, accepting only non-worsening changes) to isolate the value of accepting worse moves.
3. **Genetic algorithm.** Encode one room-period assignment per meeting. Initialize a population containing the greedy timetable and randomized candidates. Use tournament selection, course-level crossover, meeting reassignment mutation, and elitism. Apply a bounded conflict-repair procedure after crossover and mutation. Repair time counts toward the runtime budget.

Because H and S are sums of local constraint counts, moving or swapping meetings changes only a few terms. We will compute ΔE incrementally rather than rescoring the full timetable, so every method can evaluate many candidate changes within the time budget. The independent validator will recompute the full score of every reported timetable.

Our main implementation will use Python, NumPy, pandas, and Matplotlib, with pinned dependencies and recorded seeds. We will request instructor approval for any methods outside the course coverage.

**Parameter selection:** Compare a small, equal-budget set of development configurations: annealing temperatures and cooling rates, and genetic population sizes and mutation rates. Select settings using feasibility first and preference penalty second, then freeze them before evaluating held-out instances. The score weights will not be tuned to favor a method.

**Scope:** The required deliverable is the three-method comparison, plus the hill-climbing ablation, on our documented formulation. An optional extension is to parse a small selection of public timetabling instances after matching their constraints; it is not required for the core project. At proposal time, this repository contains the plan and report template; no working scheduler or experimental results are claimed.

# Evaluation Metrics

All methods will use the same instances, validator, and score. We will report:

- **Feasibility rate:** number of runs with H = 0 divided by total runs.
- **Preference penalty:** S and S/L for feasible outputs, plus the individual G, U, and R components. Failed runs remain visible and are not treated as low-penalty successes.
- **Computation:** elapsed time to the first feasible timetable, objective evaluations, and best feasible penalty at fixed time budgets. Runs that never become feasible will be recorded as failures at the cutoff.
- **Scalability and variability:** results for each course count and contention level u, with median and interquartile range across instances and seeds.

The initial evaluation will allow one second per run, including initialization and repair, on one documented machine. Each stochastic method will use three seeds per test instance; the deterministic greedy baseline will run once per instance. A preselected stratified subset will also receive five-second runs to examine improvement with additional time. Development timing will determine whether budgets need adjustment before test evaluation.

We will compare methods on matched instances and show feasibility curves, penalty distributions, and example timetable grids. Quality comparisons will also use the subset of instances where all methods find feasible outputs, alongside each method's full feasibility rate, to make differing success rates clear. We will report paired penalty differences, test them with a Wilcoxon signed-rank test on per-instance medians across seeds, and avoid claiming optimality without a proven bound.

# Ethics & Privacy

We will use synthetic instructors and cohorts and collect no personal schedules or student records. Preference weights can favor some groups, and an aggregate score may conceal one cohort receiving a particularly inconvenient timetable. We will therefore also inspect per-cohort idle time and undesirable-period assignments. Our simplified constraints omit factors such as accessibility, travel time, and individual accommodations; the resulting timetables will be research demonstrations, not recommendations for actual course allocation.

# Team Expectations

The following is a proposed working agreement for the group to review:
- Meet weekly, record decisions in the repository, and flag blockers early.
- Share implementation, experiments, and writing; each member reviews another member's work.
- Agree on the data schema, constraint definitions, and evaluation protocol before implementing separate methods.
- Use development evidence to resolve technical disagreements and document changes to scope.
- Set an internal report deadline at least two days before the official deadline.

**Proposed responsibilities:** Derrick Chen will lead the instance generator, validator, and greedy baseline; Zhen Wang will lead simulated annealing; Daira Merino will lead the genetic algorithm. All three will integrate the code, run experiments, interpret results, and write the report. These assignments can be adjusted by the team.

# Project Timeline Proposal

Dates will be aligned with the course schedule; the phases below are relative to proposal approval.

| Phase | Planned work | Lead |
|---|---|---|
| Week 1 | Finalize constraints; generate sample instances; implement validator and greedy baseline | Derrick and Zhen; all review |
| Week 2 | Implement annealing and genetic methods; verify outputs on small cases | Zhen and Daira |
| Week 3 | Tune on development instances; freeze parameters and evaluation protocol | All |
| Week 4 | Run held-out comparisons; analyze failures, runtime, and schedule quality | All |
| Before finals-week deadline | Complete report, figures, and reproduction instructions; review and export final PDF | All |

The final report will include background and citations, methods, results, discussion, and conclusion, with a link to this public repository containing source code, generation scripts, configurations, and experiment outputs. Submission will follow the official Gradescope deadline.

# Footnotes

<a name="schaerfnote"></a>1.[^](#schaerf): Schaerf, A. (1999). A survey of automated timetabling. *Artificial Intelligence Review*, 13(2), 87–127. https://doi.org/10.1023/A:1006576209967<br>
<a name="burkenote"></a>2.[^](#burke): Burke, E. K., and Petrovic, S. (2002). Recent research directions in automated timetabling. *European Journal of Operational Research*, 140(2), 266–280. https://doi.org/10.1016/S0377-2217(02)00069-3<br>
<a name="evennote"></a>3.[^](#even): Even, S., Itai, A., and Shamir, A. (1976). On the complexity of timetable and multicommodity flow problems. *SIAM Journal on Computing*, 5(4), 691–703. https://doi.org/10.1137/0205048<br>
<a name="mullernote"></a>4.[^](#muller): Müller, T. (2009). ITC2007 solver description: A hybrid approach. *Annals of Operations Research*, 172(1), 429–446. https://doi.org/10.1007/s10479-009-0644-y ([author's version](https://muller.unitime.org/itc2007.pdf))<br>
<a name="aimanote"></a>5.[^](#aima): Russell, S., and Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson. Chapter 4 (local search, simulated annealing, genetic algorithms) and Chapter 6 (constraint satisfaction, min-conflicts). https://aima.cs.berkeley.edu/<br>
<a name="mintonnote"></a>6.[^](#minton): Minton, S., Johnston, M. D., Philips, A. B., and Laird, P. (1992). Minimizing conflicts: A heuristic repair method for constraint satisfaction and scheduling problems. *Artificial Intelligence*, 58(1–3), 161–205. https://doi.org/10.1016/0004-3702(92)90007-K<br>
<a name="kirkpatricknote"></a>7.[^](#kirkpatrick): Kirkpatrick, S., Gelatt, C. D., Jr., and Vecchi, M. P. (1983). Optimization by simulated annealing. *Science*, 220(4598), 671–680. https://doi.org/10.1126/science.220.4598.671<br>
<a name="abramsonnote"></a>8.[^](#abramson): Abramson, D. (1991). Constructing school timetables using simulated annealing: Sequential and parallel algorithms. *Management Science*, 37(1), 98–113. https://doi.org/10.1287/mnsc.37.1.98<br>
<a name="hollandnote"></a>9.[^](#holland): Holland, J. H. (1992). *Adaptation in Natural and Artificial Systems* (reprint of 1975 ed.). MIT Press. https://doi.org/10.7551/mitpress/1090.001.0001<br>
<a name="lewisnote"></a>10.[^](#lewis): Lewis, R. (2008). A survey of metaheuristic-based techniques for university timetabling problems. *OR Spectrum*, 30(1), 167–190. https://doi.org/10.1007/s00291-007-0097-0